# Zero-shot Bangla Medical VQA on BanglaMedVQA: OpenMed Ministral-3B-MedVL (Real + White Blank Control)

**Protocol & Parity with `Ministral-3B`:**
- Model: `OpenMed/Ministral-3B-MedVL` (pinned revision `4829656b3f34ae88307688a3d525ceff9daff972`, `Mistral3ForConditionalGeneration`)
- Proven T4 loading: symlinks `consolidated.safetensors` -> `model.safetensors` in a temp dir (no 7.2 GB copy) and loads directly onto `cuda:0` with `attn_implementation="sdpa"` and `torch.bfloat16` (matching `RadImageNet_VQA_Ministral_3B_Instruct_MedVL.ipynb`).
- Preprocessing parity: real images are converted to RGB and aspect-preserving resized with `longest_edge <= 512` (`IMAGE_MAX_SIDE = 512`, `rgb_aspect_preserving_longest_edge_512_v1`), matching the completed `Ministral-3B` run.
- Conditions (loaded **once**, run sequentially into separate subdirectories):
  1. `real/`: original medical images (`500` questions)
  2. `white/`: solid white `512x512` RGB `(255, 255, 255)` control images (`500` questions)
- Dataset: `iiCEMAN/BanglaMedVQA` (`chest_xray_train`, `medicat_train`), frozen 500-question sample (`SAMPLE_HASH = caed7502c8c2`, 455 unique `image_id`s)
- Prompt: BanglaMedVQA Appendix C zero-shot `SYSTEM_PROMPT` (verbatim) + untranslated Bangla questions (`question_bn`)
- Decoding: Greedy (`do_sample=False`, `num_beams=1`), `seed=42`, `MAX_NEW_TOKENS=96`
- Primary metrics: Strict Exact Match (`clean()`), BERTScore F1 (`bert-base-multilingual-cased`, layer 9), BanglaMedVQA-adapted LAVE (`gpt-4.1-mini`, `temperature=0`)
- Secrets needed: `HF_TOKEN` and `OPENAI_API_KEY`.

## 1. Setup, Dependencies, and Secrets
> **Colab note:** If `%pip install` prompts for a runtime restart, click **Restart session** and then run from the next cell onward (skipping `%pip install`).

In [1]:
%pip install -q "transformers>=5.0.0,<6.0.0" "accelerate>=1.1.0" "huggingface-hub>=0.36.0" "datasets>=3.6.0" safetensors sentencepiece mistral-common scikit-learn pandas tqdm "Pillow>=10" openai

In [2]:
import csv, gc, hashlib, json, math, os, random, re, shutil, time, unicodedata
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from collections import Counter
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import torch
from PIL import Image
from datasets import load_dataset
from huggingface_hub import login, snapshot_download
from openai import OpenAI
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    Mistral3ForConditionalGeneration,
)

def get_secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        val = UserSecretsClient().get_secret(name)
        if val:
            return val
    except Exception:
        pass
    return None

HF_TOKEN = get_secret("HF_TOKEN")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("WARNING: HF_TOKEN not set; gated or rate-limited downloads may fail.")

TAG = "ministral_3b_medvl"
MODEL_ID = "OpenMed/Ministral-3B-MedVL"
MODEL_REVISION = "4829656b3f34ae88307688a3d525ceff9daff972"
CONDITIONS = ("real", "white")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DTYPE = torch.bfloat16
MAX_NEW_TOKENS = 96
BLANK_SIZE = (512, 512)
IMAGE_MAX_SIDE = 512
IMAGE_PREPROCESSING_VERSION = "rgb_aspect_preserving_longest_edge_512_v1"
DATASET = "iiCEMAN/BanglaMedVQA"
SPLITS = {"chest_xray": "chest_xray_train", "medicat": "medicat_train"}
CATEGORIES = ["abnormality", "condition", "modality", "organ", "position"]
N_PER_CELL = 50
EXPECTED_SAMPLE_HASH = "caed7502c8c2"
JUDGE_MODEL = "gpt-4.1-mini"

CWD = Path.cwd()
if (CWD / "BanglaMedVQA").is_dir():
    OUT_ROOT = CWD / "BanglaMedVQA" / "Ministral_3B_MedVL" / f"{TAG}_results"
else:
    OUT_ROOT = CWD / f"{TAG}_results"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FROZEN_CANDIDATES = [
    CWD / "BanglaMedVQA" / "Gemma" / "bnvqa_sample.csv",
    CWD.parent / "Gemma" / "bnvqa_sample.csv",
    CWD / "bnvqa_sample.csv",
]
print("OUT_ROOT:", OUT_ROOT)
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

OUT_ROOT: /content/ministral_3b_medvl_results
CUDA: True | GPU: Tesla T4


## 2. Load BanglaMedVQA and Verify the Frozen 500-Question Sample
Reproduces the seed-42 sampling and enforces `SAMPLE_HASH == "caed7502c8c2"` plus exact alignment with `BanglaMedVQA/Gemma/bnvqa_sample.csv` if present.

In [3]:
ds = {src: load_dataset(DATASET, split=split, token=HF_TOKEN) for src, split in SPLITS.items()}

frames = []
for src, d in ds.items():
    m = d.remove_columns("image").to_pandas()
    m["source"] = src
    m["src_idx"] = np.arange(len(m))
    frames.append(m)
meta = pd.concat(frames, ignore_index=True)

def yes_no(ans_en):
    s = str(ans_en).strip().lower()
    if s.startswith("yes"):
        return "yes"
    if s.startswith("no") and not s.startswith(("none", "not ")):
        return "no"
    return None

meta["yn"] = meta["answer_en"].map(yes_no)

picked = []
for src in SPLITS:
    for cat in CATEGORIES:
        pool = meta[(meta.source == src) & (meta.category == cat)]
        pool = pool.sample(frac=1, random_state=SEED).drop_duplicates("image_id")
        if cat == "abnormality":
            pool = pool[pool.yn.notna()]
            yes, no = pool[pool.yn == "yes"], pool[pool.yn == "no"]
            n_no = min(N_PER_CELL // 2, len(no))
            n_yes = N_PER_CELL - n_no
            cell = pd.concat([yes.head(n_yes), no.head(n_no)])
        else:
            cell = pool.head(N_PER_CELL)
        assert len(cell) == N_PER_CELL, f"{src}/{cat}: expected {N_PER_CELL}, got {len(cell)}"
        picked.append(cell)

sample = (
    pd.concat(picked)
    .sort_values(["source", "category", "src_idx"])
    .reset_index(drop=True)
)
sample.insert(0, "idx", np.arange(len(sample)))

ids = "|".join(f"{s}:{i}" for s, i in zip(sample.source, sample.src_idx))
SAMPLE_HASH = hashlib.md5(ids.encode()).hexdigest()[:12]
assert SAMPLE_HASH == EXPECTED_SAMPLE_HASH, f"Sample hash mismatch: {SAMPLE_HASH} != {EXPECTED_SAMPLE_HASH}"

id_cols = ["idx", "source", "src_idx", "image_id", "category"]
for cand in FROZEN_CANDIDATES:
    if cand.exists():
        frozen = pd.read_csv(cand)
        assert frozen[id_cols].astype(str).equals(sample[id_cols].astype(str)), (
            f"Sample mismatch against frozen {cand}!"
        )
        print("Verified exact match against frozen sample:", cand)
        break

sample[id_cols].to_csv(OUT_ROOT / "bnvqa_sample.csv", index=False)
print(f"Sample ready: {len(sample)} questions | {sample.image_id.nunique()} unique image_ids | hash={SAMPLE_HASH}")
print(sample.groupby(["source", "category"]).size().unstack(0))

README.md:   0%|          | 0.00/6.04k [00:00<?, ?B/s]

data/chest_xray_train-00000-of-00003.par(…): reconstructing file:   0%|          |  0.00B /  473MB            

data/chest_xray_train-00000-of-00003.par(…): downloading bytes:           |  0.00B            

data/chest_xray_train-00001-of-00003.par(…): reconstructing file:   0%|          |  0.00B /  471MB            

data/chest_xray_train-00001-of-00003.par(…): downloading bytes:           |  0.00B            

data/chest_xray_train-00002-of-00003.par(…): reconstructing file:   0%|          |  0.00B /  466MB            

... (12 outputs left out)

Generating chest_xray_test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating medicat_train split:   0%|          | 0/3500 [00:00<?, ? examples/s]

Generating medicat_test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Verified exact match against frozen sample: /content/bnvqa_sample.csv
Sample ready: 500 questions | 455 unique image_ids | hash=caed7502c8c2
source       chest_xray  medicat
category                        
abnormality          50       50
condition            50       50
modality             50       50
organ                50       50
position             50       50


## 3. Prompt, Image Conditions, Cleaning, and Formatting Diagnostics

In [4]:
# Zero-shot system prompt from the BanglaMedVQA paper (Appendix C), verbatim.
SYSTEM_PROMPT = """You are a careful, clinically grounded medical vision-language assistant.

Task: You will be given a medical image and a single question about that image (e.g., modality, organ, abnormality, condition, or the position of the condition). Your job is to look at the image and provide the exact, minimal answer to the question.

CRITICAL: You must respond with ONLY the exact answer. No explanations, no sentences, no extra words.

STRICT OUTPUT RULES:
- ONE word or short phrase only
- NO sentences or explanations
- NO "This is..." or "The image shows..."
- NO punctuation unless part of the answer"""

PROMPT_SHA256 = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()

def get_image(row, condition):
    if condition == "real":
        image = ds[row["source"]][int(row["src_idx"])]["image"].convert("RGB")
        image.thumbnail((IMAGE_MAX_SIDE, IMAGE_MAX_SIDE), Image.Resampling.LANCZOS)
        return image
    if condition == "white":
        return Image.new("RGB", BLANK_SIZE, (255, 255, 255))
    raise ValueError(f"Unknown condition: {condition}")

def clean(raw):
    """Minimal cleaning, same for every model: <answer> tag, whitespace, wrapping quotes, trailing danda/dot."""
    s = str(raw) if raw is not None and not (isinstance(raw, float) and math.isnan(raw)) else ""
    m = re.search(r"<answer>(.*?)(</answer>|$)", s, re.S)
    if m:
        s = m.group(1)
    s = s.strip().strip('"' + "'" + '\u201c\u201d\u2018\u2019`').strip()
    s = re.sub(r"[\u0964.\s]+$", "", s)
    return s

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
YES_SET = {"হ্যাঁ", "হ্যা", "হাঁ", "yes"}
NO_SET = {"না", "no"}

def norm(s):
    """Supplementary normalization (Bangla/English yes-no + unicode/punctuation normalization)."""
    s = unicodedata.normalize("NFC", str(s)).lower().translate(BN_DIGITS)
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\s+", " ", s).strip()
    if s in YES_SET:
        return "yes"
    if s in NO_SET:
        return "no"
    return s

EXPLANATORY_RE = re.compile(
    r"\b(the image (shows|demonstrates|indicates|reveals|suggests|does not)|"
    r"this (is|image)|there is|it (shows|appears|may))\b",
    re.I,
)

def extract_supplementary_answer(raw, category):
    cleaned = clean(raw)
    if not cleaned:
        return "", "empty"
    words = cleaned.split()
    if len(words) <= 4 and not EXPLANATORY_RE.search(cleaned):
        return cleaned, "exact_clean"

    m_yn = re.match(r"^\s*(yes|no|হ্যাঁ|হ্যা|হাঁ|না)(?:[\s,.:;\-\u0964]|$)", cleaned, re.I)
    if m_yn:
        return m_yn.group(1), "leading_yesno"

    first_clause = re.split(r"(?:\n+|[.\u0964]\s+)", cleaned, maxsplit=1)[0].strip()
    first_clause = re.sub(r"[\u0964.,:;\s]+$", "", first_clause)
    if first_clause and len(first_clause.split()) <= 5 and not EXPLANATORY_RE.search(first_clause):
        return first_clause, "leading_clause"

    if category == "modality":
        m_mod = re.search(
            r"\b(chest x-ray|x-ray|xray|radiograph|radiography|ct scan|computed tomography|ct|fmri|mri|ultrasound)\b",
            cleaned,
            re.I,
        )
        if m_mod:
            return m_mod.group(1), "modality_keyword"

    return "", "no_reliable_extraction"

def compute_formatting_flags(raw, cleaned):
    s = "" if raw is None else str(raw)
    has_bn = bool(re.search(r"[\u0980-\u09ff]", cleaned))
    has_latin = bool(re.search(r"[A-Za-z]", cleaned))
    n_words = len(cleaned.split()) if cleaned.strip() else 0
    is_empty = cleaned.strip() == ""
    is_verbose = (n_words > 6) or bool(EXPLANATORY_RE.search(s)) or ("\n" in s.strip())
    is_exact_compliant = (not is_empty) and (not is_verbose) and (n_words <= 4)
    return {
        "is_empty": is_empty,
        "is_english_only": has_latin and not has_bn,
        "has_bangla": has_bn,
        "is_long": n_words > 6,
        "is_verbose": is_verbose,
        "is_exact_compliant": is_exact_compliant,
        "raw_char_len": len(s),
        "cleaned_word_count": n_words,
    }

## 4. Proven T4 Model Loading (`Mistral3ForConditionalGeneration` + `sdpa`) and Prompt Serialization
- Uses the exact symlink loading pattern proven in `RadImageNet_VQA_Ministral_3B_Instruct_MedVL.ipynb` (zero CPU offloading, zero extra disk copy, fast `sdpa` attention, full `cuda:0` placement in ~7.2 GB VRAM).

In [5]:
class NumericsGuard(LogitsProcessor):
    def __init__(self):
        self.invalid = False

    def __call__(self, input_ids, scores):
        self.invalid |= bool(
            (torch.isnan(scores) | torch.isposinf(scores)).any()
            or torch.isneginf(scores).all(dim=-1).any()
        )
        return scores


def load_model():
    if not torch.cuda.is_available():
        raise RuntimeError("Select a CUDA GPU runtime (Colab T4) before starting inference.")

    config_files = [
        "config.json", "generation_config.json", "processor_config.json",
        "preprocessor_config.json", "tokenizer.json", "tokenizer_config.json",
        "special_tokens_map.json", "chat_template.jinja",
    ]
    snapshot = Path(snapshot_download(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=HF_TOKEN,
        allow_patterns=config_files + ["consolidated.safetensors"],
    ))
    processor = AutoProcessor.from_pretrained(snapshot, local_files_only=True)

    with TemporaryDirectory(prefix="ministral-medvl-") as local:
        ckpt = Path(local)
        for fn in ("config.json", "generation_config.json"):
            if (snapshot / fn).exists():
                (ckpt / fn).symlink_to(snapshot / fn)
        (ckpt / "model.safetensors").symlink_to(snapshot / "consolidated.safetensors")

        model, report = Mistral3ForConditionalGeneration.from_pretrained(
            ckpt,
            local_files_only=True,
            use_safetensors=True,
            dtype=DTYPE,
            device_map={"": "cuda:0"},
            attn_implementation="sdpa",
            output_loading_info=True,
        )

    faults = {k: report[k] for k in ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs") if report.get(k)}
    if faults:
        raise RuntimeError(f"Incomplete weight load for {MODEL_ID}: {faults}")

    meta_params = [n for n, p in model.named_parameters() if p.device.type == "meta"]
    if meta_params:
        raise RuntimeError(f"Unloaded meta parameters remain: {meta_params[:8]}")

    model.generation_config.max_length = None
    model.eval()
    print(f"Loaded {TAG} ({MODEL_REVISION[:8]}) on {torch.cuda.get_device_name(0)} | {DTYPE} | sdpa")
    return model, processor


def build_ministral_messages(image, question_bn):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question_bn}]},
    ]


def render_ministral_prompt(processor, image, question_bn):
    messages = build_ministral_messages(image, question_bn)
    rendered = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if SYSTEM_PROMPT not in rendered:
        raise RuntimeError("Serialized prompt omitted SYSTEM_PROMPT! Refusing silent fallback.")
    if "[IMG]" not in rendered or question_bn not in rendered:
        raise RuntimeError("Serialized prompt missing [IMG] or question_bn!")
    if not (rendered.index(SYSTEM_PROMPT) < rendered.index("[IMG]") < rendered.index(question_bn)):
        raise RuntimeError("Serialized prompt token ordering invalid!")
    return messages, rendered


@torch.inference_mode()
def generate_one(model, processor, image, question_bn):
    messages, _ = render_ministral_prompt(processor, image, question_bn)
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    if "pixel_values" not in inputs or not (inputs["input_ids"] == model.config.image_token_index).any():
        raise RuntimeError("Processor did not encode the image and its placeholder.")
    inputs = inputs.to(model.device, dtype=DTYPE)
    guard = NumericsGuard()
    tok = processor.tokenizer
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id,
        logits_processor=LogitsProcessorList([guard]),
    )
    tokens = out[0, inputs["input_ids"].shape[1]:].tolist()
    raw = tok.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return raw, len(tokens), guard.invalid

## 5. Independent Resume Validation & Sequential Inference (Real -> White)
- Loads the model **once** and runs both conditions sequentially (no separate smoke-test reload).
- Validates `run_settings.json` and any existing CSV row-by-row before resuming; checkpoints atomically every 25 items.

In [6]:
def write_csv_atomic(path, df):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)

IMMUTABLE_KEYS = [
    "dataset", "sample_hash", "sample_size", "system_prompt_sha256",
    "model_tag", "model_id", "model_revision",
    "condition", "seed", "dtype", "max_new_tokens", "blank_size",
    "image_max_side", "image_preprocessing_version",
]

def prepare_condition(condition):
    cond_dir = OUT_ROOT / condition
    cond_dir.mkdir(parents=True, exist_ok=True)
    out_csv = cond_dir / f"bnvqa_results_{TAG}.csv"
    settings_json = cond_dir / "run_settings.json"

    current_settings = {
        "dataset": DATASET,
        "splits": SPLITS,
        "sample_hash": SAMPLE_HASH,
        "sample_size": len(sample),
        "unique_image_ids": int(sample.image_id.nunique()),
        "system_prompt_sha256": PROMPT_SHA256,
        "model_tag": TAG,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "condition": condition,
        "seed": SEED,
        "dtype": str(DTYPE),
        "max_new_tokens": MAX_NEW_TOKENS,
        "blank_size": list(BLANK_SIZE) if condition == "white" else None,
        "image_max_side": IMAGE_MAX_SIDE,
        "image_preprocessing_version": IMAGE_PREPROCESSING_VERSION,
        "judge_model": JUDGE_MODEL,
        "out_dir": str(cond_dir),
    }
    if settings_json.exists():
        saved = json.loads(settings_json.read_text(encoding="utf-8"))
        for k in IMMUTABLE_KEYS:
            if saved.get(k) != current_settings.get(k):
                raise ValueError(f"Resume rejected ({condition}): run_settings.json mismatch on '{k}'")
    else:
        current_settings["torch"] = str(torch.__version__)
        current_settings["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        settings_json.write_text(json.dumps(current_settings, indent=2, sort_keys=True), encoding="utf-8")

    if out_csv.exists():
        done = pd.read_csv(out_csv)
        done["predicted_answer"] = done["predicted_answer"].fillna("").astype(str)
        req_cols = ["idx", "image_id", "source", "category", "predicted_answer", "raw_output"]
        missing = [c for c in req_cols if c not in done.columns]
        if missing:
            raise ValueError(f"Resume rejected ({condition}): missing columns {missing}")
        if len(done) > len(sample) or not done["idx"].is_unique:
            raise ValueError(f"Resume rejected ({condition}): invalid row count or duplicate idx")
        for col in ["idx", "image_id", "source", "category"]:
            if done[col].astype(str).tolist() != sample[col].iloc[:len(done)].astype(str).tolist():
                raise ValueError(f"Resume rejected ({condition}): '{col}' mismatch against frozen sample")
        rows = done.to_dict("records")
        seen = set(done["idx"].astype(int))
        print(f"[{condition}] Validated existing {len(rows)}/{len(sample)} rows in {out_csv}")
    else:
        rows = []
        seen = set()

    sample[["idx", "source", "src_idx", "image_id", "category"]].to_csv(cond_dir / "bnvqa_sample.csv", index=False)
    return cond_dir, out_csv, rows, seen

state = {cond: prepare_condition(cond) for cond in CONDITIONS}
need_inference = any(len(state[cond][2]) < len(sample) for cond in CONDITIONS)

if not need_inference:
    print("Both real and white conditions already have all 500 validated rows; skipping model loading.")
else:
    model, processor = load_model()
    _, sample_rendered = render_ministral_prompt(processor, get_image(sample.iloc[0], "real"), sample.iloc[0]["question_bn"])
    print("Prompt serialization verified (len =", len(sample_rendered), "chars).")
    try:
        for cond in CONDITIONS:
            cond_dir, out_csv, rows, seen = state[cond]
            if len(rows) == len(sample):
                print(f"[{cond}] Already complete (500/500), skipping.")
                continue
            for _, r in tqdm(sample.iterrows(), total=len(sample), desc=f"{TAG} [{cond}]"):
                idx_val = int(r["idx"])
                if idx_val in seen:
                    continue
                img = get_image(r, cond)
                raw, n_tok, invalid = generate_one(model, processor, img, r["question_bn"])
                cleaned = "" if invalid else clean(raw)
                ext_ans, ext_method = extract_supplementary_answer(raw, r["category"])
                flags = compute_formatting_flags(raw, cleaned)
                rows.append({
                    "idx": idx_val,
                    "image_id": r["image_id"],
                    "source": r["source"],
                    "category": r["category"],
                    "question": r["question_bn"],
                    "true_answer": r["answer_bn"],
                    "predicted_answer": cleaned,
                    "raw_output": raw,
                    "extracted_answer": ext_ans,
                    "extraction_method": ext_method,
                    "gen_tokens": n_tok,
                    "logits_invalid": bool(invalid),
                    **flags,
                })
                if len(rows) % 25 == 0:
                    write_csv_atomic(out_csv, pd.DataFrame(rows).sort_values("idx").reset_index(drop=True))
                if len(rows) % 50 == 0:
                    gc.collect()
                    torch.cuda.empty_cache()
            df = pd.DataFrame(rows).sort_values("idx").reset_index(drop=True)
            assert df["idx"].astype(int).tolist() == sample["idx"].astype(int).tolist()
            write_csv_atomic(out_csv, df)
            print(f"Saved {len(df)} rows -> {out_csv}")
    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

[transformers] The tokenizer you are loading from '/root/.cache/huggingface/hub/models--OpenMed--Ministral-3B-MedVL/snapshots/4829656b3f34ae88307688a3d525ceff9daff972' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loaded ministral_3b_medvl (4829656b) on Tesla T4 | torch.bfloat16 | sdpa
Prompt serialization verified (len = 674 chars).


ministral_3b_medvl [real]:   0%|          | 0/500 [00:00<?, ?it/s]

Saved 500 rows -> /content/ministral_3b_medvl_results/real/bnvqa_results_ministral_3b_medvl.csv


ministral_3b_medvl [white]:   0%|          | 0/500 [00:00<?, ?it/s]

Saved 500 rows -> /content/ministral_3b_medvl_results/white/bnvqa_results_ministral_3b_medvl.csv


## 6. Primary & Exploratory Metrics (Strict EM, Norm EM, Exploratory Extracted EM, BERTScore F1)

In [7]:
@torch.no_grad()
def bertscore_f1(cands, refs, model_name="bert-base-multilingual-cased", layer=9, bs=32, tok=None, mdl=None):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = tok or AutoTokenizer.from_pretrained(model_name)
    if mdl is None:
        mdl = AutoModel.from_pretrained(model_name)
        mdl.encoder.layer = mdl.encoder.layer[:layer]
    mdl = mdl.to(dev).eval()

    def embed(texts):
        out = []
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=512).to(dev)
            h = torch.nn.functional.normalize(mdl(**enc).last_hidden_state.float(), dim=-1)
            for j in range(h.size(0)):
                n = int(enc["attention_mask"][j].sum())
                out.append(h[j, :n].cpu())
        return out

    C, R = embed(list(cands)), embed(list(refs))
    f1 = []
    for c, r in zip(C, R):
        if len(c) <= 2 or len(r) <= 2:
            f1.append(0.0)
            continue
        sim = r @ c.T
        rec = sim.max(1).values[1:-1].mean()
        prec = sim.max(0).values[1:-1].mean()
        f1.append(float(2 * prec * rec / (prec + rec)))
    return torch.tensor(f1)

_bs_tok = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
_bs_mdl = AutoModel.from_pretrained("bert-base-multilingual-cased")
_bs_mdl.encoder.layer = _bs_mdl.encoder.layer[:9]

for cond in CONDITIONS:
    out_csv = OUT_ROOT / cond / f"bnvqa_results_{TAG}.csv"
    df = pd.read_csv(out_csv).sort_values("idx").reset_index(drop=True)
    df["predicted_answer"] = df["predicted_answer"].fillna("").astype(str)
    df["extracted_answer"] = df["extracted_answer"].fillna("").astype(str)
    df["true_answer"] = df["true_answer"].astype(str)

    df["correct"] = df.true_answer.map(clean) == df.predicted_answer
    df["correct_norm"] = df.true_answer.map(norm) == df.predicted_answer.map(norm)
    df["correct_extracted_exploratory"] = df.true_answer.map(clean) == df.extracted_answer.map(clean)
    df["correct_extracted_norm_exploratory"] = df.true_answer.map(norm) == df.extracted_answer.map(norm)

    if "bertscore_f1" not in df or df["bertscore_f1"].isna().any():
        preds = [p if p.strip() else " " for p in df.predicted_answer]
        df["bertscore_f1"] = bertscore_f1(preds, df.true_answer.tolist(), tok=_bs_tok, mdl=_bs_mdl).numpy()
        df.loc[df.predicted_answer.str.strip() == "", "bertscore_f1"] = 0.0

    write_csv_atomic(out_csv, df)
    print(
        f"[{TAG} | {cond}] Strict EM={df.correct.mean()*100:.2f}% | "
        f"NormEM={df.correct_norm.mean()*100:.2f}% | "
        f"Exploratory ExtNorm={df.correct_extracted_norm_exploratory.mean()*100:.2f}% | "
        f"BERTScore={df.bertscore_f1.mean()*100:.2f}%"
    )

del _bs_mdl, _bs_tok
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[ministral_3b_medvl | real] Strict EM=11.60% | NormEM=11.80% | Exploratory ExtNorm=11.80% | BERTScore=75.08%
[ministral_3b_medvl | white] Strict EM=10.00% | NormEM=10.00% | Exploratory ExtNorm=10.00% | BERTScore=73.89%


## 7. LAVE Judge Evaluation (`gpt-4.1-mini`, Paper Appendix C Prompt)

In [8]:
LAVE_SYSTEM = (
    "You are a strict evaluator. "
    "Compare each prediction with its reference. "
    "Return ONLY a valid JSON array of floats between 0 and 1. "
    "Each float represents similarity between reference and prediction. "
    "If a reference answer is missing, treat it as 'No'. "
    "Do NOT return any text outside the JSON."
)

def lave_score(client, ref, pred, retries=3):
    msg = f"Reference: {ref}\nPrediction: {pred}"
    last = None
    for _ in range(retries):
        try:
            r = client.chat.completions.create(
                model=JUDGE_MODEL,
                temperature=0,
                messages=[
                    {"role": "system", "content": LAVE_SYSTEM},
                    {"role": "user", "content": msg},
                ],
            )
            out = r.choices[0].message.content
            val = json.loads(re.search(r"\[.*?\]", out, re.S).group())[0]
            return min(max(float(val), 0.0), 1.0)
        except Exception as e:
            last = e
            time.sleep(2)
    print("LAVE failed:", type(last).__name__)
    return None

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not set; skipping LAVE calls. Set key and re-run this cell.")
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    for cond in CONDITIONS:
        out_csv = OUT_ROOT / cond / f"bnvqa_results_{TAG}.csv"
        df = pd.read_csv(out_csv)
        if "lave" not in df:
            df["lave"] = np.nan
        todo = df.index[df.lave.isna()]
        for n, i in enumerate(tqdm(todo, desc=f"LAVE {TAG} [{cond}]")):
            df.at[i, "lave"] = lave_score(client, df.at[i, "true_answer"], df.at[i, "predicted_answer"])
            if (n + 1) % 25 == 0:
                write_csv_atomic(out_csv, df)
        write_csv_atomic(out_csv, df)
        print(f"[{TAG} | {cond}] LAVE={df.lave.mean()*100:.2f}% | failures={int(df.lave.isna().sum())}")

LAVE ministral_3b_medvl [real]:   0%|          | 0/500 [00:00<?, ?it/s]

[ministral_3b_medvl | real] LAVE=29.59% | failures=0


LAVE ministral_3b_medvl [white]:   0%|          | 0/500 [00:00<?, ?it/s]

[ministral_3b_medvl | white] LAVE=14.32% | failures=0


## 8. Summary Tables, Formatting Diagnostics, Stratified Extraction Audit, and Zip Export

In [9]:
def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

def summarize(g):
    n, k = len(g), int(g.correct.sum())
    lo, hi = wilson(k, n)
    return pd.Series({
        "n": n,
        "accuracy": k / n,
        "acc_ci_low": lo,
        "acc_ci_high": hi,
        "accuracy_norm": g.correct_norm.mean(),
        "accuracy_extracted_exploratory": g.correct_extracted_exploratory.mean(),
        "accuracy_extracted_norm_exploratory": g.correct_extracted_norm_exploratory.mean(),
        "bertscore": g.bertscore_f1.mean(),
        "lave": g.lave.mean() if "lave" in g else np.nan,
        "lave_failures": int(g.lave.isna().sum()) if "lave" in g else n,
    })

for cond in CONDITIONS:
    cond_dir = OUT_ROOT / cond
    df = pd.read_csv(cond_dir / f"bnvqa_results_{TAG}.csv")
    df["predicted_answer"] = df["predicted_answer"].fillna("").astype(str)
    df["true_answer"] = df["true_answer"].astype(str)

    by_cell = df.groupby(["source", "category"]).apply(summarize).reset_index()
    by_src = df.groupby("source").apply(summarize).reset_index().assign(category="all")
    overall = summarize(df).to_frame().T.assign(source="all", category="all")
    avg = by_src[["accuracy", "accuracy_norm", "accuracy_extracted_exploratory", "accuracy_extracted_norm_exploratory", "bertscore", "lave"]].mean().to_frame().T.assign(
        source="avg_of_sources", category="all", n=len(df)
    )
    scores_df = pd.concat([by_cell, by_src, overall, avg], ignore_index=True)
    scores_df.insert(0, "condition", cond)
    scores_df.insert(0, "model", TAG)
    write_csv_atomic(cond_dir / f"bnvqa_scores_{TAG}.csv", scores_df)

    fmt_cols = ["is_empty", "is_english_only", "has_bangla", "is_long", "is_verbose", "is_exact_compliant", "raw_char_len", "cleaned_word_count"]
    fmt_cat = df.groupby("category")[fmt_cols].mean().reset_index()
    fmt_all = df[fmt_cols].mean().to_frame().T.assign(category="all")
    write_csv_atomic(cond_dir / "formatting_diagnostics.csv", pd.concat([fmt_cat, fmt_all], ignore_index=True))

    ab = df[df.category == "abnormality"].copy()
    ab["true_yn"] = ab.true_answer.map(norm)
    ab["pred_yn"] = ab.predicted_answer.map(norm).where(lambda s: s.isin(["yes", "no"]), "other")
    rep_txt = classification_report(ab.true_yn, ab.pred_yn, labels=["yes", "no"], zero_division=0)
    cm_df = pd.DataFrame(
        confusion_matrix(ab.true_yn, ab.pred_yn, labels=["yes", "no", "other"]),
        index=["true_yes", "true_no", "true_other"],
        columns=["pred_yes", "pred_no", "pred_other"],
    )
    (cond_dir / "abnormality_report.txt").write_text(rep_txt + "\n" + cm_df.to_string(), encoding="utf-8")

    audit_rows = [grp.sample(min(3, len(grp)), random_state=SEED) for _, grp in df.groupby(["source", "category"])]
    audit_df = pd.concat(audit_rows).sort_values("idx")[
        ["idx", "source", "category", "true_answer", "predicted_answer", "extracted_answer", "extraction_method", "is_verbose", "raw_output"]
    ]
    write_csv_atomic(cond_dir / "extraction_inspection_sample.csv", audit_df)

archive = shutil.make_archive(str(OUT_ROOT.parent / "ministral_3b_medvl_bnvqa_real_and_white"), "zip", OUT_ROOT)
print("Saved real & white outputs and zip:", archive)
print("IMPORTANT: Colab /content storage is temporary across runtime resets. Download the ZIP before ending your session.")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(archive)))

Saved real & white outputs and zip: /content/ministral_3b_medvl_bnvqa_real_and_white.zip
IMPORTANT: Colab /content storage is temporary across runtime resets. Download the ZIP before ending your session.


/tmp/ipykernel_703/2658856398.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_cell = df.groupby(["source", "category"]).apply(summarize).reset_index()
/tmp/ipykernel_703/2658856398.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_src = df.groupby("source").apply(summarize).reset_index().assign(category="all")
/tmp/ipykernel_703/2658856398.py:32: DeprecationWarning: DataFrameGroupBy.apply oper

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>